In [1]:
# Optimización de hiperparámetros para MLP (hipoxia) 
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


2025-08-30 19:45:54.339092: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# ---------- 1) Cargar datos ----------
datos = pd.read_csv("caracteristicas.csv")
X = datos.drop(columns=["hipoxia", "marca_tiempo", "estacion"], errors="ignore")
y = datos["hipoxia"].astype(int)
# train/test (para reporte final). Dentro de train haremos sub-train/validación
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
# sub-train / validación (solo para la búsqueda)
X_sub, X_val, y_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.20, stratify=y_train, random_state=42
)
# Escalado SIN fuga: fit en sub-train, transformar val y test
scaler = StandardScaler()
X_sub_s = scaler.fit_transform(X_sub)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)
# Pesos por clase (por si hipoxia es minoritaria)
clases = np.unique(y_sub)
pesos = compute_class_weight("balanced", classes=clases, y=y_sub)
pesos_clase = {int(c): float(w) for c, w in zip(clases, pesos)}
print("Pesos por clase:", pesos_clase)


Pesos por clase: {0: 0.5153448275862069, 1: 16.792134831460675}


In [4]:
# ---------- 2) Definir constructor del modelo ----------
def crear_modelo(n1=64, n2=32, lr=1e-3, dropout=0.0, input_dim=None):
    m = Sequential([
        Dense(n1, activation="relu", input_shape=(input_dim,)),
        Dropout(dropout),
        Dense(n2, activation="relu"),
        Dropout(dropout),
        Dense(1, activation="sigmoid")
    ])
    m.compile(optimizer=Adam(learning_rate=lr),
              loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return m

In [5]:
# ---------- 3) Cuadrícula pequeña de hiperparámetros ----------
grid = {
    "n1": [64, 128],
    "n2": [32, 64],
    "lr": [1e-3, 5e-4],
    "batch": [64, 128, 256],
    "dropout": [0.0, 0.2]
}

In [6]:
# ---------- 4) Búsqueda manual (rápida) ----------
resultados = []
mejor_auc = -1.0
mejores = None
mejor_modelo = None

for n1 in grid["n1"]:
    for n2 in grid["n2"]:
        for lr in grid["lr"]:
            for batch in grid["batch"]:
                for dr in grid["dropout"]:
                    print(f"Probando: n1={n1}, n2={n2}, lr={lr}, batch={batch}, dropout={dr}")
                    modelo = crear_modelo(n1=n1, n2=n2, lr=lr, dropout=dr, input_dim=X_sub_s.shape[1])
                    es = EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True)
                    hist = modelo.fit(
                        X_sub_s, y_sub,
                        validation_data=(X_val_s, y_val),
                        epochs=30, batch_size=batch,
                        class_weight=pesos_clase,
                        callbacks=[es],
                        verbose=0
                    )
                    # Mejor AUC en validación de este entrenamiento
                    val_auc = max(hist.history["val_auc"])
                    val_acc = max(hist.history["val_accuracy"])
                    resultados.append({
                        "n1": n1, "n2": n2, "lr": lr, "batch": batch, "dropout": dr,
                        "val_auc": float(val_auc), "val_acc": float(val_acc),
                        "epochs_usadas": len(hist.history["val_auc"])
                    })
                    if val_auc > mejor_auc:
                        mejor_auc = val_auc
                        mejores = {"n1": n1, "n2": n2, "lr": lr, "batch": batch, "dropout": dr}
                        mejor_modelo = modelo

# Guardar tabla de resultados ordenada
df_res = pd.DataFrame(resultados).sort_values("val_auc", ascending=False)
df_res.to_csv("mlp_tuning_resultados.csv", index=False)
print("\nTOP 5 por AUC en validación:")
print(df_res.head(5))

print("\nMejores hiperparámetros:", mejores)
print(f"Mejor AUC (validación): {mejor_auc:.3f}")

Probando: n1=64, n2=32, lr=0.001, batch=64, dropout=0.0


/home/luisrueda/PROYECTO-IA/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-08-30 19:49:00.435795: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Probando: n1=64, n2=32, lr=0.001, batch=64, dropout=0.2
Probando: n1=64, n2=32, lr=0.001, batch=128, dropout=0.0
Probando: n1=64, n2=32, lr=0.001, batch=128, dropout=0.2
Probando: n1=64, n2=32, lr=0.001, batch=256, dropout=0.0
Probando: n1=64, n2=32, lr=0.001, batch=256, dropout=0.2
Probando: n1=64, n2=32, lr=0.0005, batch=64, dropout=0.0
Probando: n1=64, n2=32, lr=0.0005, batch=64, dropout=0.2
Probando: n1=64, n2=32, lr=0.0005, batch=128, dropout=0.0
Probando: n1=64, n2=32, lr=0.0005, batch=128, dropout=0.2
Probando: n1=64, n2=32, lr=0.0005, batch=256, dropout=0.0
Probando: n1=64, n2=32, lr=0.0005, batch=256, dropout=0.2
Probando: n1=64, n2=64, lr=0.001, batch=64, dropout=0.0
Probando: n1=64, n2=64, lr=0.001, batch=64, dropout=0.2
Probando: n1=64, n2=64, lr=0.001, batch=128, dropout=0.0
Probando: n1=64, n2=64, lr=0.001, batch=128, dropout=0.2
Probando: n1=64, n2=64, lr=0.001, batch=256, dropout=0.0
Probando: n1=64, n2=64, lr=0.001, batch=256, dropout=0.2
Probando: n1=64, n2=64, lr=0.0

In [ ]:
# ---------- 5) Evaluación final en TEST con el mejor modelo ----------
y_prob = mejor_modelo.predict(X_test_s).ravel()
y_pred = (y_prob >= 0.5).astype(int)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
auc  = roc_auc_score(y_test, y_prob)

print("\n=== MÉTRICAS EN TEST (mejor MLP) ===")
print(f"Exactitud: {acc:.3f}  |  Precisión: {prec:.3f}  |  Recall: {rec:.3f}  |  F1: {f1:.3f}  |  AUC: {auc:.3f}")
# (Opcional) Guardar el modelo y el escalador
mejor_modelo.save("mlp_hipoxia_mejor.keras")
import joblib; joblib.dump(scaler, "mlp_hipoxia_scaler.joblib")
print("\nGuardado: mlp_hipoxia_mejor.keras, mlp_hipoxia_scaler.joblib y mlp_tuning_resultados.csv")


In [ ]:
#Qué hace
#Divide train/test y dentro de train crea sub-train/validación para el tuning (evita fuga).
#Prueba combinaciones pequeñas de: neuronas (n1, n2), learning rate, batch size y dropout.
#Selecciona el mejor por AUC en validación.
#Reporta métricas en TEST del mejor MLP y guarda una tabla con todos los intentos.